# SNI-MRENet fail-fast dan resumable
Notebook validation-only untuk 21 kelas instance crop SNI. Setiap stage berada pada sel terpisah, checkpoint ditulis langsung ke Google Drive, dan test tidak pernah dibuka. Training memakai AMP FP16 + channels-last dengan statistik HBP tetap FP32; epoch, batch, data, dan metode tidak berubah. Jalankan stage berikutnya hanya bila stage sebelumnya PASS.

In [ ]:
SEEDS = [42]
REPO_REF = 'agent/sni-instance-crops'
DRIVE_DATA_FOLDER = 'coffee-sni-instance-crop-v1'
DRIVE_RESULT_FOLDER = 'sni-mrenet-v1'
EVALUATION_SPLIT = 'val'

In [ ]:
# 1/7 SETUP REPOSITORY, DRIVE, DAN GPU
import csv, json, shutil, subprocess, sys, tarfile, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
assert SEEDS == [42] and EVALUATION_SPLIT == 'val'
repo = Path('/content/coffee-bean-classification')
repo_url = 'https://github.com/ediprin/coffee-bean-classification.git'
def run(command, cwd=None):
    command = [str(item) for item in command]
    print('\n$', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)
if (repo / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_REF], cwd=repo)
    run(['git', 'checkout', REPO_REF], cwd=repo)
    run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=repo)
else:
    if repo.exists(): shutil.rmtree(repo)
    run(['git', 'clone', '--branch', REPO_REF, repo_url, repo])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo])
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU melalui Runtime > Change runtime type.'
print('GPU :', torch.cuda.get_device_name(0))
print('REPO:', repo)
print('OPTIMASI: AMP FP16 + channels_last + non_blocking; statistik HBP tetap FP32')

In [ ]:
# 2/7 PULIHKAN DATASET DARI SHARD GOOGLE DRIVE
data_root = Path('/content/sni-instance-crops')
drive_data = Path('/content/drive/MyDrive') / DRIVE_DATA_FOLDER
shard_root = drive_data / 'shards'
if not (data_root / 'audit.json').is_file():
    complete = drive_data / 'complete.json'
    shards = sorted(shard_root.glob('crop_shard_*.tar'))
    assert complete.is_file() and shards, (
        'Backup dataset belum lengkap. Jalankan notebook '
        'sni_instance_crop_preparation_colab.ipynb terlebih dahulu.'
    )
    data_root.mkdir(parents=True, exist_ok=True)
    for index, shard in enumerate(shards, 1):
        with tarfile.open(shard, 'r') as archive:
            archive.extractall(data_root, filter='data')
        if index % 5 == 0 or index == len(shards):
            print(f'RESTORE {index}/{len(shards)}', flush=True)
    for name in ('audit.json', 'manifest.csv'):
        source = drive_data / name
        assert source.is_file(), f'Backup metadata hilang: {source}'
        shutil.copy2(source, data_root / name)
    allowed = set()
    with (data_root / 'manifest.csv').open(newline='', encoding='utf-8') as handle:
        allowed = {row['crop_path'] for row in csv.DictReader(handle)}
    for path in (data_root / 'source').rglob('*.jpg'):
        if path.relative_to(data_root).as_posix() not in allowed:
            path.unlink()
audit = json.loads((data_root / 'audit.json').read_text())
assert audit['status'] == 'complete'
assert audit['num_classes'] == 21
assert audit['output_crops'] == 31074
assert audit['generated_cross_split_identity_count'] == 0
assert audit['test_locked'] is True
assert all((data_root / f'source/{split}').is_dir() for split in ('train', 'val', 'test'))
output_root = Path('/content/drive/MyDrive') / DRIVE_RESULT_FOLDER
output_root.mkdir(parents=True, exist_ok=True)
print('DATASET:', data_root)
print('CROPS  :', audit['output_crops'])
print('OUTPUT :', output_root)
print('TEST LOCKED:', audit['test_locked'])

In [ ]:
# 3/7 HELPER STAGE, HEARTBEAT, DAN PUTUSAN
def stage_report(stage):
    path = output_root / f'val_reports/sni_mrenet_{stage}.json'
    assert path.is_file(), f'Report belum tersedia: {path}'
    report = json.loads(path.read_text())
    print(f'\n=== PUTUSAN {stage.upper()} ===')
    for name, row in report['decisions'].items():
        print(name, row['decision'], row['criteria'])
    print('FINAL:', report['final_decision'])
    print('Test dibuka:', report['test_opened'])
    return report
def run_stage(stage, codes):
    command = [
        sys.executable, '-u', '-m',
        'bilinear_lmmd.experiments.run_sni_mrenet_screening',
        '--data-root', str(data_root),
        '--output-root', str(output_root),
        '--seeds', *map(str, SEEDS),
        '--stage', stage, '--evaluation-split', EVALUATION_SPLIT,
    ]
    print('MENJALANKAN:', ' '.join(command), flush=True)
    process = subprocess.Popen(command, cwd=repo)
    started = time.monotonic()
    while process.poll() is None:
        status = []
        for code in codes:
            history = output_root / f'outputs/{code}_seed42/history.json'
            if history.is_file():
                try: status.append(f'{code}={len(json.loads(history.read_text()))}/50')
                except Exception: status.append(f'{code}=saving')
        print(f'[{stage.upper()} {(time.monotonic()-started)/60:.1f} menit]', ', '.join(status) if status else 'inisialisasi', flush=True)
        time.sleep(60)
    assert process.wait() == 0, f'Stage {stage} gagal; baca traceback di atas.'
    return stage_report(stage)

In [ ]:
# 4/7 STAGE A - BASELINE VS BACKBONE MULTIRESOLUSI
backbone_report = run_stage('backbone', ('SNIB0', 'SNIB1'))
if backbone_report['final_decision'] != 'PASS':
    print('STOP: multiresolusi tidak lolos. Jangan jalankan sel berikutnya.')

In [ ]:
# 5/7 STAGE B - ONTOLOGY EXPERT DENGAN FIRST-ORDER CONTROL
backbone_report = stage_report('backbone')
assert backbone_report['final_decision'] == 'PASS', 'STOP: stage backbone FAIL.'
ontology_report = run_stage('ontology', ('SNIB2',))
if ontology_report['final_decision'] != 'PASS':
    print('STOP: ontology expert tidak lolos. Jangan jalankan sel berikutnya.')

In [ ]:
# 6/7 STAGE C - HBP SELEKTIF, CAPACITY-MATCHED DENGAN SNIB2
ontology_report = stage_report('ontology')
assert ontology_report['final_decision'] == 'PASS', 'STOP: stage ontology FAIL.'
bilinear_report = run_stage('bilinear', ('SNIB3',))
if bilinear_report['final_decision'] != 'PASS':
    print('STOP: selective HBP tidak lolos. Jangan buka test atau seed lain.')

In [ ]:
# 7/7 RINGKASAN FINAL SCREENING - TIDAK MENJALANKAN TRAINING BARU
reports = {stage: stage_report(stage) for stage in ('backbone', 'ontology', 'bilinear')}
passed = all(report['final_decision'] == 'PASS' for report in reports.values())
print('\nSCREENING SEED 42:', 'PASS' if passed else 'FAIL')
print('Checkpoint persisten:', output_root / 'outputs')
print('Test tetap terkunci. Seed 123/2026 hanya dibuat setelah hasil ini dinilai.')

## Aturan penggunaan
Jangan memakai Run all tanpa mengawasi keputusan setiap stage. Jika runtime reset, jalankan ulang sel 1-3 lalu sel stage yang terakhir; engine memakai `--resume` dan checkpoint berada di Drive. Jangan menjalankan seed 123/2026 atau test dari notebook ini.